# Analyse av nedbygd areal per kommune (2023–2025)

Denne notebooken analyserer CSV-en du eksporterte fra Google Earth Engine (GEE) med areal **nedbygd/built** per kommune (`kid`) for sommerperioder i 2023, 2024 og 2025.

**Du trenger:**
- Python 3.9+
- `pandas`, `numpy`, `matplotlib` (og valgfritt `seaborn`)

## Hvordan bruke
1. Last ned CSV-en fra Google Drive.
2. Legg CSV-filen i samme mappe som denne notebooken **eller** oppdater `CSV_PATH` i cellen under.
3. Kjør cellene fra topp til bunn.

> Tips: Hvis du har flere CSV-er (f.eks. south/north), finnes en egen seksjon for å slå de sammen.


In [ ]:
# --- Imports og grunnoppsett ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Valgfritt, men gir penere plots
try:
    import seaborn as sns
    sns.set_context('talk')
    sns.set_style('whitegrid')
except Exception:
    sns = None

plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['figure.dpi'] = 110

print('Klar!')

## 1) Last inn CSV
CSV-en bør inneholde kolonner som ligner: 
- `kid`
- `built_km2_2023`, `built_km2_2024`, `built_km2_2025`
- (valgfritt) `delta_km2_23_25`, `newbuilt_km2_23_25`

Notebooken forsøker å gjenkjenne/rydde opp kolonnenavn hvis de avviker litt.


In [ ]:
# --- Sett filnavn her ---
CSV_PATH = Path('built_per_kid_2023_2025_summer_500m_south.csv')  # <-- endre hvis nødvendig

if not CSV_PATH.exists():
    raise FileNotFoundError(f'Fant ikke {CSV_PATH}. Legg CSV-en i samme mappe eller oppdater CSV_PATH.')

df_raw = pd.read_csv(CSV_PATH)
print('Rader:', len(df_raw), ' Kolonner:', len(df_raw.columns))
df_raw.head()

In [ ]:
# Kolonneoversikt
df_raw.columns.tolist()

## 2) Standardiser kolonner og datatyper
GEE kan noen ganger eksportere tall med vitenskapelig notasjon eller som tekst.
Vi konverterer relevante kolonner til numerisk og lager noen derived metrics.


In [ ]:
df = df_raw.copy()

# --- Heuristikk for å finne riktig ID-kolonne ---
candidate_id = [c for c in df.columns if c.lower() in ['kid', 'kommune', 'kommunenummer', 'kommunenum', 'knr']]
if 'kid' not in df.columns and candidate_id:
    df = df.rename(columns={candidate_id[0]: 'kid'})

if 'kid' not in df.columns:
    raise ValueError('Fant ikke kommune-id kolonne. Forventet kolonne `kid` (eller lignende).')

df['kid'] = pd.to_numeric(df['kid'], errors='coerce').astype('Int64')

# --- Finn built-kolonner automatisk ---
def find_col(keys):
    keys = [k.lower() for k in keys]
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in keys):
            return c
    return None

c23 = find_col(['built', '2023']) or find_col(['km2', '2023'])
c24 = find_col(['built', '2024']) or find_col(['km2', '2024'])
c25 = find_col(['built', '2025']) or find_col(['km2', '2025'])

# Hvis de allerede heter built_km2_YYYY, holder det
rename_map = {}
if c23 and c23 != 'built_km2_2023': rename_map[c23] = 'built_km2_2023'
if c24 and c24 != 'built_km2_2024': rename_map[c24] = 'built_km2_2024'
if c25 and c25 != 'built_km2_2025': rename_map[c25] = 'built_km2_2025'
df = df.rename(columns=rename_map)

# Delta/newbuilt om de finnes
dcol = find_col(['delta', '23', '25']) or find_col(['delta', '2023', '2025'])
ncol = find_col(['newbuilt', '23', '25']) or find_col(['new', 'built', '23', '25'])
if dcol and dcol != 'delta_km2_23_25':
    df = df.rename(columns={dcol: 'delta_km2_23_25'})
if ncol and ncol != 'newbuilt_km2_23_25':
    df = df.rename(columns={ncol: 'newbuilt_km2_23_25'})

# Konverter numeriske kolonner (tåler komma som desimalskilletegn)
num_cols = [c for c in ['built_km2_2023','built_km2_2024','built_km2_2025','delta_km2_23_25','newbuilt_km2_23_25'] if c in df.columns]
for c in num_cols:
    df[c] = (df[c].astype(str)
               .str.replace(',', '.', regex=False)
               .replace({'nan': np.nan, 'None': np.nan}))
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Lag delta hvis ikke finnes
if 'delta_km2_23_25' not in df.columns and all(c in df.columns for c in ['built_km2_2023','built_km2_2025']):
    df['delta_km2_23_25'] = df['built_km2_2025'] - df['built_km2_2023']

# Lag prosentendring
if all(c in df.columns for c in ['built_km2_2023','built_km2_2025']):
    df['pct_change_23_25'] = np.where(df['built_km2_2023']>0,
                                     100*(df['built_km2_2025']-df['built_km2_2023'])/df['built_km2_2023'],
                                     np.nan)

# Rydd duplikater
df = df.drop_duplicates(subset=['kid']).sort_values('kid')

df.head()

In [ ]:
# Hurtig kvalitetsrapport
display(df.describe(include='all'))
print('Manglende per kolonne:')
print(df.isna().sum().sort_values(ascending=False).head(10))

## 3) Aggregerte nøkkeltall (totalt for regionen din)
Dette gir en rask sanity check: total built km² per år og total endring.


In [ ]:
totals = {}
for y in [2023, 2024, 2025]:
    c = f'built_km2_{y}'
    if c in df.columns:
        totals[y] = df[c].sum(skipna=True)

totals_series = pd.Series(totals, name='total_built_km2')
totals_series

In [ ]:
# Plot total built per år
plt.figure()
totals_series.sort_index().plot(kind='line', marker='o')
plt.title('Totalt nedbygd areal (built) i regionen – sommerkompositt')
plt.xlabel('År')
plt.ylabel('km²')
plt.tight_layout()
plt.show()

## 4) Rangering og endring per kommune
Vi ser på hvilke kommuner som øker mest (absolutt og relativt).


In [ ]:
# Topplister
top_abs = df.nlargest(20, 'delta_km2_23_25')[['kid','delta_km2_23_25','built_km2_2023','built_km2_2025']].copy()
bottom_abs = df.nsmallest(20, 'delta_km2_23_25')[['kid','delta_km2_23_25','built_km2_2023','built_km2_2025']].copy()

top_abs.head(), bottom_abs.head()

In [ ]:
# Plot: Top 20 økning (km²)
plt.figure(figsize=(12, 8))
tmp = top_abs.sort_values('delta_km2_23_25')
plt.barh(tmp['kid'].astype(str), tmp['delta_km2_23_25'])
plt.title('Top 20 kommuner – størst økning i built (km²) fra 2023 til 2025')
plt.xlabel('Δ built (km²)')
plt.ylabel('kid')
plt.tight_layout()
plt.show()

In [ ]:
# Plot: Top 20 reduksjon (km²) – kan skyldes klassestøy
plt.figure(figsize=(12, 8))
tmp = bottom_abs.sort_values('delta_km2_23_25')
plt.barh(tmp['kid'].astype(str), tmp['delta_km2_23_25'], color='tab:red')
plt.title('Top 20 kommuner – størst reduksjon i built (km²) fra 2023 til 2025')
plt.xlabel('Δ built (km²)')
plt.ylabel('kid')
plt.tight_layout()
plt.show()

## 5) Distribusjoner og usikkerhet
Negative endringer kan komme av klassestøy/sky/snø-effekter, selv med sommer-vindu.
Dette plotter fordelingen av endringer.


In [ ]:
plt.figure()
data = df['delta_km2_23_25'].dropna()
plt.hist(data, bins=40, color='tab:blue', alpha=0.8)
plt.axvline(0, color='black', linewidth=1)
plt.title('Fordeling: Δ built (km²) 2023→2025')
plt.xlabel('Δ built (km²)')
plt.ylabel('Antall kommuner')
plt.tight_layout()
plt.show()

In [ ]:
# Robust visning med percentiler
q = df['delta_km2_23_25'].quantile([0.01,0.05,0.5,0.95,0.99])
q

## 6) Sammenheng: hvor mye som allerede er built vs. vekst
Et interessant plot er om kommuner med mye built i 2023 også vokser mest i absolutte tall.


In [ ]:
plt.figure()
x = df['built_km2_2023']
y = df['delta_km2_23_25']
plt.scatter(x, y, alpha=0.6)
plt.axhline(0, color='black', linewidth=1)
plt.title('Built 2023 vs. endring 2023→2025')
plt.xlabel('built (km²) i 2023')
plt.ylabel('Δ built (km²) 2023→2025')
plt.tight_layout()
plt.show()

# Samme med log-skala på x (ofte mer informativt)
plt.figure()
plt.scatter(x.replace(0, np.nan), y, alpha=0.6)
plt.xscale('log')
plt.axhline(0, color='black', linewidth=1)
plt.title('Built 2023 vs. endring 2023→2025 (log x-akse)')
plt.xlabel('built (km²) i 2023 (log)')
plt.ylabel('Δ built (km²) 2023→2025')
plt.tight_layout()
plt.show()

## 7) Konsentrasjon: hvor mye av built-arealet ligger i få kommuner?
Vi lager en Lorenz-kurve og beregner Gini-koeffisient for fordelingen av built i 2025.


In [ ]:
def gini(array):
    x = np.asarray(array, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)
    # Gini: 1 + (1/n) - 2 * sum(cumx)/(n*cumx[-1])
    return 1 + 1/n - 2*np.sum(cumx)/(n*cumx[-1])

vals = df['built_km2_2025'].fillna(0).values
g = gini(vals)
g

In [ ]:
# Lorenz-kurve
x = np.sort(df['built_km2_2025'].fillna(0).values)
cum = np.cumsum(x)
cum = np.insert(cum, 0, 0)
cum = cum / cum[-1] if cum[-1] > 0 else cum
p = np.linspace(0, 1, len(cum))

plt.figure()
plt.plot(p, cum, label='Lorenz')
plt.plot([0,1], [0,1], '--', color='gray', label='Lik fordeling')
plt.title(f'Lorenz-kurve for built (2025) – Gini = {g:.2f}')
plt.xlabel('Andel kommuner')
plt.ylabel('Kumulativ andel built (km²)')
plt.legend()
plt.tight_layout()
plt.show()

## 8) Hvem står for mesteparten av økningen? (kumulativt bidrag)
Dette viser hvor stor del av total Δ 2023→2025 som kommer fra de største bidragsyterne.


In [ ]:
d = df[['kid','delta_km2_23_25']].dropna().sort_values('delta_km2_23_25', ascending=False)
d['delta_pos'] = d['delta_km2_23_25'].clip(lower=0)
total_pos = d['delta_pos'].sum()
d['cum_share'] = d['delta_pos'].cumsum() / total_pos if total_pos > 0 else np.nan

plt.figure()
plt.plot(np.arange(1, len(d)+1), d['cum_share'])
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
plt.axhline(0.8, color='gray', linestyle='--', linewidth=1)
plt.title('Kumulativ andel av total positiv økning (Δ built)')
plt.xlabel('Antall kommuner (sortert etter Δ synkende)')
plt.ylabel('Kumulativ andel av total Δ (kun positive bidrag)')
plt.tight_layout()
plt.show()

# Hvor mange kommuner trengs for 50% og 80%?
n50 = int((d['cum_share'] >= 0.5).idxmax() + 1) if total_pos > 0 else None
n80 = int((d['cum_share'] >= 0.8).idxmax() + 1) if total_pos > 0 else None
print('Kommuner for 50% av økningen:', n50)
print('Kommuner for 80% av økningen:', n80)

## 9) Korrelasjoner mellom år
En enkel korrelasjonsmatrise kan gi en rask følelse for stabilitet mellom år.


In [ ]:
cols = [c for c in ['built_km2_2023','built_km2_2024','built_km2_2025','delta_km2_23_25'] if c in df.columns]
corr = df[cols].corr()
corr

In [ ]:
# Heatmap hvis seaborn er tilgjengelig
if sns is not None:
    plt.figure(figsize=(8,6))
    sns.heatmap(corr, annot=True, cmap='viridis', vmin=-1, vmax=1)
    plt.title('Korrelasjon mellom built-areal (km²) og delta')
    plt.tight_layout()
    plt.show()
else:
    print('Seaborn ikke tilgjengelig – hoppet over heatmap.')

## 10) (Valgfritt) Slå sammen flere CSV-er (f.eks. south + north)
Hvis du har kjørt to exports (eller flere) og vil summe dem per `kid`, bruk funksjonen under.


In [ ]:
def merge_csvs_sum_by_kid(paths):
    dfs = []
    for p in paths:
        d = pd.read_csv(p)
        if 'kid' not in d.columns:
            # heuristikk
            cid = [c for c in d.columns if c.lower() in ['kid','kommunenum','kommunenummer','knr']]
            if cid:
                d = d.rename(columns={cid[0]:'kid'})
        dfs.append(d)
    all_df = pd.concat(dfs, ignore_index=True)
    # summer alle numeriske kolonner per kid
    num = all_df.select_dtypes(include=[np.number]).columns
    out = all_df.groupby('kid', as_index=False)[list(num)].sum()
    return out

# Eksempel:
# merged = merge_csvs_sum_by_kid(['south.csv','north.csv'])
# merged.head()

## 11) Eksporter resultater
Vi lagrer en ryddet versjon av tabellen og topplister som CSV.


In [ ]:
out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)

df.to_csv(out_dir / 'built_kid_cleaned.csv', index=False)
top_abs.to_csv(out_dir / 'top20_increase_delta_km2.csv', index=False)
bottom_abs.to_csv(out_dir / 'top20_decrease_delta_km2.csv', index=False)
print('Skrev filer til:', out_dir.resolve())

---
### Videre forslag (hvis du vil forbedre faglig kvalitet)
- Legg inn kommunenavn ved å merge mot en lookup-tabell (CSV) med `kid → navn`.
- Kjør sensitivitetsanalyse på terskel (0.5 / 0.6 / 0.7).
- Se på utvalgte kommuner og visualiser `built` med et kartverktøy (folium/geopandas) hvis du har geometrien lokalt.
